In [ ]:
# ============================================================
# 1.라이브러리 설치
# ============================================================


# !pip install -q torch torchvision torchaudio pandas pillow opencv-python matplotlib


In [ ]:
# ============================================================
# 2. Google Drive 마운트
# ============================================================
# 학습 결과(.pt, .csv)를 런타임 종료 후에도 보존하려면 Drive에 저장하는 것이 편합니다.

from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# ============================================================
# 3. 라이브러리 import
# ============================================================
# os / random / time: 경로 처리, 셔플, 시간 측정
# namedtuple: ResNet 설정 묶기
# cv2 / PIL: 이미지 읽기
# matplotlib: 시각화
# pandas: CSV 저장
# torch 계열: 모델 구성 및 학습

import os
import random
import time
from collections import namedtuple

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

# 현재 런타임에서 GPU를 사용할 수 있으면 cuda,
# 아니면 cpu를 사용합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


In [ ]:
# ============================================================
# 4. 실험 설정값
# ============================================================

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
OUTPUT_DIM = 2          # 고양이/개 2분류
EPOCHS = 10
LEARNING_RATE = 1e-7
NUM_WORKERS = 2         # Colab에서는 보통 2~4 정도가 무난
PIN_MEMORY = torch.cuda.is_available()

# Google Drive 내 기본 경로
DRIVE_ROOT = '/content/drive/MyDrive'
CHAP_ROOT = os.path.join(DRIVE_ROOT, 'chap06')
DATA_ROOT = os.path.join(CHAP_ROOT, 'data', 'dogs-vs-cats')
CAT_DIR = os.path.join(DATA_ROOT, 'Cat')
DOG_DIR = os.path.join(DATA_ROOT, 'Dog')

# 결과 저장 경로
SAVE_DIR = os.path.join(CHAP_ROOT, 'data')
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'ResNet-model.pt')
CSV_SAVE_PATH = os.path.join(SAVE_DIR, 'ResNet.csv')

os.makedirs(SAVE_DIR, exist_ok=True)

# 재현성을 높이기 위한 시드 고정
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('DATA_ROOT      =', DATA_ROOT)
print('MODEL_SAVE_PATH=', MODEL_SAVE_PATH)
print('CSV_SAVE_PATH  =', CSV_SAVE_PATH)


In [ ]:
# ============================================================
# 5. 데이터 폴더 존재 여부 점검
# ============================================================

# 먼저 Cat / Dog 폴더가 실제로 있는지 확인합니다.

print('CAT_DIR exists:', os.path.isdir(CAT_DIR), CAT_DIR)
print('DOG_DIR exists:', os.path.isdir(DOG_DIR), DOG_DIR)

if not os.path.isdir(CAT_DIR) or not os.path.isdir(DOG_DIR):
    raise FileNotFoundError(
        'Cat 또는 Dog 폴더를 찾지 못했습니다. '        'Drive 경로와 폴더 구조를 먼저 확인하세요.'
    )


In [ ]:
# ============================================================
# 6. 정규화 파라미터
# ============================================================
# ImageNet 사전학습 계열에서 자주 사용하는 mean/std 값입니다.
# 입력 이미지를 정규화해서 학습 안정성을 높여줍니다.

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


In [ ]:
# ============================================================
# 7. 이미지 전처리 클래스
# ============================================================
# train 단계와 val 단계에서 서로 다른 전처리를 적용합니다.
#
# train:
#   - RandomResizedCrop: 이미지를 랜덤하게 자르고 크기 맞춤
#   - RandomHorizontalFlip: 좌우 반전 증강
#   - ToTensor: PIL 이미지를 텐서로 변환
#   - Normalize: 평균/표준편차 기준 정규화
#
# val:
#   - Resize -> CenterCrop: 평가 시에는 랜덤성이 없도록 중앙 기준으로 통일
#   - ToTensor / Normalize

class ImageTransform:
    def __init__(self, resize, mean, std):
        self.data_transform = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(resize, scale=(0.5, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ]),
            'val': transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(resize),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ]),
        }

    def __call__(self, img, phase):
        return self.data_transform[phase](img)


In [ ]:
# ============================================================
# 8. 사용자 정의 Dataset 클래스
# ============================================================
# 파일 경로 목록을 받아서,
#   1) 이미지를 읽고
#   2) 전처리를 적용한 뒤
#   3) 라벨(고양이=0, 개=1)을 붙여서 반환합니다.
#
# 파일명 예시:
#   cat.0.jpg -> 라벨 0
#   dog.12.jpg -> 라벨 1

class DogvsCatDataset(Dataset):
    def __init__(self, file_list, transform=None, phase='train'):
        self.file_list = file_list
        self.transform = transform
        self.phase = phase

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]

        # RGB 3채널로 강제 변환하여 입력 형태를 통일합니다.
        img = Image.open(img_path).convert('RGB')

        # phase(train / val)에 맞는 전처리를 수행합니다.
        img_transformed = self.transform(img, self.phase)

        # 파일명에서 라벨 이름(cat / dog)을 추출합니다.
        label_name = os.path.basename(img_path).split('.')[0].lower()

        if label_name == 'dog':
            label = 1
        elif label_name == 'cat':
            label = 0
        else:
            raise ValueError(f'Unknown label in path: {img_path}')

        return img_transformed, label


In [ ]:
# ============================================================
# 9. 파일 경로 수집 + 유효 이미지 필터링 + train/val/test 분할
# ============================================================
# 교재 코드 흐름을 따라,
# Cat / Dog 폴더의 파일을 모은 뒤
# cv2로 실제 읽히는 파일만 남기고
# 셔플 후 train / val / test로 나눕니다.

cat_images_filepaths = sorted([os.path.join(CAT_DIR, f) for f in os.listdir(CAT_DIR)])
dog_images_filepaths = sorted([os.path.join(DOG_DIR, f) for f in os.listdir(DOG_DIR)])

images_filepaths = [*cat_images_filepaths, *dog_images_filepaths]

# 손상 파일이나 읽기 실패 파일 제거
correct_images_filepaths = [p for p in images_filepaths if cv2.imread(p) is not None]

random.seed(SEED)
random.shuffle(correct_images_filepaths)

# 교재 OCR 결과를 반영한 분할 방식
train_images_filepaths = correct_images_filepaths[:400]
val_images_filepaths   = correct_images_filepaths[400:-10]
test_images_filepaths  = correct_images_filepaths[-10:]

print('전체 이미지 수  :', len(images_filepaths))
print('유효 이미지 수  :', len(correct_images_filepaths))
print('train 이미지 수 :', len(train_images_filepaths))
print('val 이미지 수   :', len(val_images_filepaths))
print('test 이미지 수  :', len(test_images_filepaths))


In [ ]:
# ============================================================
# 10. Dataset / DataLoader 구성
# ============================================================
# DataLoader는 미니배치를 자동으로 만들어 주고,
# shuffle=True 설정을 통해 학습 데이터를 매 epoch마다 섞어줍니다.

train_dataset = DogvsCatDataset(
    train_images_filepaths,
    transform=ImageTransform(IMAGE_SIZE, MEAN, STD),
    phase='train'
)

val_dataset = DogvsCatDataset(
    val_images_filepaths,
    transform=ImageTransform(IMAGE_SIZE, MEAN, STD),
    phase='val'
)

train_iterator = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

valid_iterator = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print('train batch 수 =', len(train_iterator))
print('valid batch 수 =', len(valid_iterator))


In [ ]:
# ============================================================
# 11. 배치 형태 확인
# ============================================================
# 실제로 DataLoader가 어떤 모양의 텐서를 뽑는지 확인합니다.
# 일반적으로 [배치크기, 채널수, 높이, 너비] 형태가 나옵니다.

sample_inputs, sample_labels = next(iter(train_iterator))
print('입력 텐서 shape =', sample_inputs.shape)
print('라벨 텐서 shape =', sample_labels.shape)
print('라벨 예시       =', sample_labels[:10])


In [ ]:
# ============================================================
# 12. ResNet BasicBlock
# ============================================================
# ResNet18 / ResNet34 계열에서 사용하는 기본 residual block입니다.
#
# 핵심 아이디어:
#   출력 = F(x) + x
# 즉, 입력 x를 지름길(skip connection)로 더해 줌으로써
# 깊은 네트워크에서도 학습이 더 잘 되도록 돕습니다.

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # downsample은 입력 x와 출력 F(x)의 shape이 다를 때
        # 1x1 convolution으로 차원을 맞추는 역할을 합니다.
        if downsample:
            conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
            bn = nn.BatchNorm2d(out_channels)
            downsample = nn.Sequential(conv, bn)
        else:
            downsample = None

        self.downsample = downsample

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        if self.downsample is not None:
            identity = self.downsample(identity)

        x = x + identity
        x = self.relu(x)
        return x


In [ ]:
# ============================================================
# 13. ResNet Bottleneck
# ============================================================
# ResNet50 / 101 / 152에서 사용하는 block입니다.
# 1x1 -> 3x3 -> 1x1 구조로 되어 있으며,
# 연산량과 채널 수를 효율적으로 다루기 위해 사용됩니다.

class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(
            out_channels,
            self.expansion * out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(self.expansion * out_channels)
        self.relu = nn.ReLU(inplace=True)

        if downsample:
            conv = nn.Conv2d(
                in_channels,
                self.expansion * out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
            bn = nn.BatchNorm2d(self.expansion * out_channels)
            downsample = nn.Sequential(conv, bn)
        else:
            downsample = None

        self.downsample = downsample

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        x = self.conv3(x)
        x = self.bn3(x)

        if self.downsample is not None:
            identity = self.downsample(identity)

        x = x + identity
        x = self.relu(x)
        return x


In [ ]:
# ============================================================
# 14. ResNet 본체 정의
# ============================================================
# 전체 흐름:
#   입력 이미지
#     -> 초기 Conv/BN/ReLU/MaxPool
#     -> layer1 ~ layer4 (residual block 묶음)
#     -> AdaptiveAvgPool
#     -> 펼치기(flatten)
#     -> FC 분류기
#
# forward는 (logits, feature_vector)를 함께 반환하도록 만들었습니다.
# logits: 최종 분류 점수
# feature_vector: FC 직전 특징 벡터

class ResNet(nn.Module):
    def __init__(self, config, output_dim, zero_init_residual=False):
        super().__init__()

        block, n_blocks, channels = config
        self.in_channels = channels[0]

        assert len(n_blocks) == len(channels) == 4

        self.conv1 = nn.Conv2d(3, self.in_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self.get_resnet_layer(block, n_blocks[0], channels[0])
        self.layer2 = self.get_resnet_layer(block, n_blocks[1], channels[1], stride=2)
        self.layer3 = self.get_resnet_layer(block, n_blocks[2], channels[2], stride=2)
        self.layer4 = self.get_resnet_layer(block, n_blocks[3], channels[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(self.in_channels, output_dim)

        # zero_init_residual=True로 주면 residual branch의 마지막 BN을 0으로 초기화하여
        # block이 처음엔 identity mapping처럼 동작하도록 돕는 기법입니다.
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def get_resnet_layer(self, block, n_blocks, channels, stride=1):
        layers = []

        # 입력 채널 수와 출력 채널 수(expansion 포함)가 다르면
        # skip connection 차원 맞춤이 필요합니다.
        downsample = self.in_channels != block.expansion * channels or stride != 1

        layers.append(block(self.in_channels, channels, stride, downsample))

        for _ in range(1, n_blocks):
            layers.append(block(block.expansion * channels, channels))

        self.in_channels = block.expansion * channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        h = x.view(x.shape[0], -1)
        logits = self.fc(h)
        return logits, h


In [ ]:
# ============================================================
# 15. ResNet 설정값 모음
# ============================================================
# 어떤 block을 몇 개 쌓을지 정의합니다.
# 현재 실습에서는 resnet50_config를 사용합니다.

ResNetConfig = namedtuple('ResNetConfig', ['block', 'n_blocks', 'channels'])

resnet18_config = ResNetConfig(
    block=BasicBlock,
    n_blocks=[2, 2, 2, 2],
    channels=[64, 128, 256, 512],
)

resnet34_config = ResNetConfig(
    block=BasicBlock,
    n_blocks=[3, 4, 6, 3],
    channels=[64, 128, 256, 512],
)

resnet50_config = ResNetConfig(
    block=Bottleneck,
    n_blocks=[3, 4, 6, 3],
    channels=[64, 128, 256, 512],
)

resnet101_config = ResNetConfig(
    block=Bottleneck,
    n_blocks=[3, 4, 23, 3],
    channels=[64, 128, 256, 512],
)

resnet152_config = ResNetConfig(
    block=Bottleneck,
    n_blocks=[3, 8, 36, 3],
    channels=[64, 128, 256, 512],
)


In [ ]:
# ============================================================
# 16. 정확도 계산 함수
# ============================================================
# 교재 OCR 결과에는 Top-1 / Top-5 정확도 계산 흐름이 보였습니다.
# 하지만 현재 문제는 2분류(cat, dog)이므로 strict한 의미의 Top-5는 불가능합니다.
# 따라서 아래 함수는 k가 클래스 수보다 크면 자동으로 클래스 수에 맞춰 줄입니다.
#
# 예:
#   클래스 수가 2개일 때
#   top-5 요청 -> 실제로는 top-2로 계산

def calculate_topk_accuracy(y_pred, y, k=5):
    with torch.no_grad():
        batch_size = y.shape[0]
        effective_k = min(k, y_pred.shape[1])

        _, top_pred = y_pred.topk(effective_k, dim=1)
        top_pred = top_pred.t()
        correct = top_pred.eq(y.view(1, -1).expand_as(top_pred))

        correct_1 = correct[:1].reshape(-1).float().sum(0, keepdim=True)
        correct_k = correct[:effective_k].reshape(-1).float().sum(0, keepdim=True)

        acc_1 = correct_1 / batch_size
        acc_k = correct_k / batch_size

    return acc_1, acc_k


In [ ]:
# ============================================================
# 17. 학습 / 평가 / 시간 측정 함수
# ============================================================
# train():
#   - 모델을 학습 모드로 전환
#   - 배치마다 순전파 -> loss 계산 -> 역전파 -> optimizer.step()
#
# evaluate():
#   - 모델을 평가 모드로 전환
#   - gradient 계산 없이 성능만 측정
#
# epoch_time():
#   - epoch 1회에 걸린 시간을 분/초로 환산

def train(model, iterator, optimizer, criterion, scheduler, device):
    epoch_loss = 0
    epoch_acc_1 = 0
    epoch_acc_k = 0

    model.train()

    for x, y in iterator:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred, _ = model(x)
        loss = criterion(y_pred, y)

        acc_1, acc_k = calculate_topk_accuracy(y_pred, y, k=5)

        loss.backward()
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        epoch_loss += loss.item()
        epoch_acc_1 += acc_1.item()
        epoch_acc_k += acc_k.item()

    epoch_loss /= len(iterator)
    epoch_acc_1 /= len(iterator)
    epoch_acc_k /= len(iterator)

    return epoch_loss, epoch_acc_1, epoch_acc_k


def evaluate(model, iterator, criterion, device):
    epoch_loss = 0
    epoch_acc_1 = 0
    epoch_acc_k = 0

    model.eval()

    with torch.no_grad():
        for x, y in iterator:
            x = x.to(device)
            y = y.to(device)

            y_pred, _ = model(x)
            loss = criterion(y_pred, y)

            acc_1, acc_k = calculate_topk_accuracy(y_pred, y, k=5)

            epoch_loss += loss.item()
            epoch_acc_1 += acc_1.item()
            epoch_acc_k += acc_k.item()

    epoch_loss /= len(iterator)
    epoch_acc_1 /= len(iterator)
    epoch_acc_k /= len(iterator)

    return epoch_loss, epoch_acc_1, epoch_acc_k


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


In [ ]:
# ============================================================
# 18. 모델 / optimizer / loss 정의
# ============================================================
# 현재는 교재 흐름에 맞춰 직접 구현한 ResNet50 구조를 사용합니다.
# 출력 차원은 2 (cat / dog) 입니다.

model = ResNet(resnet50_config, OUTPUT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss().to(device)

# 교재 OCR에서 scheduler 존재 흔적은 있었지만 명확하지 않아,
# 기본값은 None으로 두었습니다.
scheduler = None

print(model)


In [ ]:
# ============================================================
# 19. 학습 루프
# ============================================================
# 검증 손실(valid loss)이 가장 낮은 시점의 모델을 저장합니다.
#
# 주의:
# - 데이터 수가 적으면 epoch별 변동이 큽니다.
# - Colab 무료 GPU에서는 학습 시간이 런타임 상황에 따라 달라질 수 있습니다.

best_valid_loss = float('inf')

for epoch in range(EPOCHS):
    start_time = time.monotonic()

    train_loss, train_acc_1, train_acc_k = train(
        model, train_iterator, optimizer, criterion, scheduler, device
    )
    valid_loss, valid_acc_1, valid_acc_k = evaluate(
        model, valid_iterator, criterion, device
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(
        f'	Train Loss: {train_loss:.3f} | '
        f'Train Acc @1: {train_acc_1*100:6.2f}% | '
        f'Train Acc @k: {train_acc_k*100:6.2f}%'
    )
    print(
        f'	Valid Loss: {valid_loss:.3f} | '
        f'Valid Acc @1: {valid_acc_1*100:6.2f}% | '
        f'Valid Acc @k: {valid_acc_k*100:6.2f}%'
    )

print('최적 모델 저장 위치:', MODEL_SAVE_PATH)


In [ ]:
# ============================================================
# 20. 저장된 최적 모델 다시 불러오기 (선택)
# ============================================================
# 추론 전에 가장 성능이 좋았던 가중치를 다시 로드하고 싶다면 실행합니다.

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.to(device)
model.eval()
print('best model loaded!')


In [ ]:
# ============================================================
# 21. 테스트 이미지 예측 후 CSV 저장
# ============================================================
# 테스트 이미지 각각에 대해 'dog 클래스일 확률'을 저장합니다.
# 결과 CSV 형식:
#   id,label
#   0,0.1234
#   1,0.9812
#   ...
#
# 여기서 id는 파일명 예: dog.123.jpg / cat.123.jpg 의 가운데 숫자를 사용합니다.

def predict_testset_to_csv(model, test_images_filepaths, size, mean, std, output_csv_path, device):
    id_list = []
    pred_list = []

    transform = ImageTransform(size, mean, std)
    model.eval()

    with torch.no_grad():
        for test_path in test_images_filepaths:
            img = Image.open(test_path).convert('RGB')

            # 예: dog.123.jpg -> 123 추출
            parts = os.path.basename(test_path).split('.')
            file_id = parts[1] if len(parts) > 1 else parts[0]

            img = transform(img, phase='val')
            img = img.unsqueeze(0).to(device)

            outputs, _ = model(img)

            # softmax 후 [:, 1]은 'dog 클래스 확률'이라고 해석할 수 있습니다.
            preds = F.softmax(outputs, dim=1)[:, 1].tolist()

            id_list.append(file_id)
            pred_list.append(preds[0])

    res = pd.DataFrame({'id': id_list, 'label': pred_list})
    res.sort_values(by='id', inplace=True)
    res.reset_index(drop=True, inplace=True)
    res.to_csv(output_csv_path, index=False)
    return res


res = predict_testset_to_csv(
    model=model,
    test_images_filepaths=test_images_filepaths,
    size=IMAGE_SIZE,
    mean=MEAN,
    std=STD,
    output_csv_path=CSV_SAVE_PATH,
    device=device,
)

print(res.head(10))
print('CSV 저장 완료:', CSV_SAVE_PATH)


In [ ]:
# ============================================================
# 22. 예측 결과 시각화
# ============================================================
# 테스트 이미지 몇 장을 보여 주고,
# 모델 예측 확률을 바탕으로 cat / dog 라벨을 붙여서 시각화합니다.

def display_image_grid(images_filepaths, res, cols=5):
    class_name = {0: 'cat', 1: 'dog'}

    n_images = len(images_filepaths)
    rows = (n_images + cols - 1) // cols

    figure, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(15, 3 * rows))
    if rows == 1:
        axes = axes if hasattr(axes, '__len__') else [axes]
    axes = axes.ravel()

    for i, image_filepath in enumerate(images_filepaths):
        image = cv2.imread(image_filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        parts = os.path.basename(image_filepath).split('.')
        file_id = parts[1] if len(parts) > 1 else parts[0]

        label_prob = float(res.loc[res['id'] == file_id, 'label'].values[0])
        pred_class = 1 if label_prob > 0.5 else 0

        axes[i].imshow(image)
        axes[i].set_title(f"pred: {class_name[pred_class]}
prob(dog)={label_prob:.3f}")
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


display_image_grid(test_images_filepaths, res, cols=5)
